# Whiteboard Digitisation Pipeline — Full Demo

**Three-stage ML pipeline:** Whiteboard photo → Object Detection → Math OCR → Structured LaTeX

| Stage | Model | Purpose |
|-------|-------|---------|
| **Detection** | YOLOv8-nano (fine-tuned) | Detect `whiteboard` + `equation` regions |
| **Post-processing** | Bbox merge + spatial filter | Merge fragmented equations, suppress ceiling FPs |
| **OCR** | pix2tex → TrOCR → PaddleOCR | Convert equation crops to LaTeX |

**Runtime:** Go to **Runtime → Change runtime type → GPU (T4)** before running.

### Preparation (run locally before uploading)

```bash
cd data/raw
zip -r composite.zip composite/
```
Upload `composite.zip` **and** a few test whiteboard photos to `My Drive/mla-proj/`

## 1. Environment Setup

In [ ]:
# Check GPU availability — you should see a Tesla T4 or better
!nvidia-smi

In [ ]:
!pip install -q ultralytics>=8.0 "pix2tex[gui]" transformers paddlepaddle paddleocr

In [ ]:
import torch
from pathlib import Path

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU      : {torch.cuda.get_device_name(0)}")
    print(f"VRAM     : {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 2. Mount Google Drive & Load Dataset

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# ── Configure paths ──────────────────────────────────────────────────────
# Edit this if your zip is in a different Drive location
DRIVE_ZIP = Path("/content/drive/MyDrive/mla-proj/composite.zip")
DATA_ROOT = Path("/content/dataset")          # fast local disk on Colab
RUNS_DIR  = Path("/content/runs/detect")      # training outputs (local)
DRIVE_OUT = Path("/content/drive/MyDrive/mla-proj/runs")  # persist to Drive

assert DRIVE_ZIP.exists(), (
    f"Dataset zip not found at {DRIVE_ZIP}\n"
    "Upload composite.zip to your Google Drive and update DRIVE_ZIP above."
)

In [ ]:
# Unzip to Colab local disk (much faster I/O than reading from Drive)
import zipfile, shutil

if DATA_ROOT.exists():
    shutil.rmtree(DATA_ROOT)

DATA_ROOT.mkdir(parents=True)
print(f"Extracting {DRIVE_ZIP.name} to {DATA_ROOT} ...")

with zipfile.ZipFile(DRIVE_ZIP, "r") as zf:
    zf.extractall(DATA_ROOT)

# The zip contains composite/{train,valid}/{images,labels}
COMPOSITE_DIR = DATA_ROOT / "composite"
if not COMPOSITE_DIR.exists():
    # In case the zip root IS the content directly
    COMPOSITE_DIR = DATA_ROOT

print(f"Train images : {len(list((COMPOSITE_DIR / 'train' / 'images').iterdir()))}")
print(f"Valid images : {len(list((COMPOSITE_DIR / 'valid' / 'images').iterdir()))}")

## 3. Create Dataset YAML

The `dataset_v2.yaml` used locally has an absolute path that won't work on Colab. We regenerate it here pointing at the Colab-local extraction path.

In [ ]:
import yaml

YAML_PATH = Path("/content/dataset_v2.yaml")

dataset_config = {
    "path": str(COMPOSITE_DIR.resolve()),
    "train": "train/images",
    "val": "valid/images",
    "nc": 2,
    "names": ["equation", "whiteboard"],
}

YAML_PATH.write_text(yaml.dump(dataset_config, default_flow_style=False))
print(f"Written: {YAML_PATH}")
print(YAML_PATH.read_text())

## 4. Inspect Dataset Samples

Sanity check: visualise a few training images with their labels to confirm the data loaded correctly.

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
import random

CLASS_NAMES = ["equation", "whiteboard"]
COLORS = [(255, 0, 0), (0, 0, 255)]  # blue for eq, red for wb (BGR)

def draw_labels(img_path, lbl_path):
    """Draw YOLO labels on an image for visual inspection."""
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]

    if lbl_path.exists():
        for line in lbl_path.read_text().splitlines():
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            cls_id = int(parts[0])
            cx, cy, bw, bh = [float(v) for v in parts[1:5]]
            x1 = int((cx - bw / 2) * w)
            y1 = int((cy - bh / 2) * h)
            x2 = int((cx + bw / 2) * w)
            y2 = int((cy + bh / 2) * h)
            color = COLORS[cls_id] if cls_id < len(COLORS) else (128, 128, 128)
            cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
            cv2.putText(img, CLASS_NAMES[cls_id], (x1, max(y1 - 5, 10)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
    return img

# Pick 4 random composited training images
train_imgs = sorted((COMPOSITE_DIR / "train" / "images").glob("comp_*"))
samples = random.sample(train_imgs, min(4, len(train_imgs)))

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, img_path in zip(axes.flat, samples):
    lbl_path = COMPOSITE_DIR / "train" / "labels" / (img_path.stem + ".txt")
    ax.imshow(draw_labels(img_path, lbl_path))
    ax.set_title(img_path.name, fontsize=9)
    ax.axis("off")
plt.suptitle("Training Samples — blue=equation, red=whiteboard", fontsize=13)
plt.tight_layout()
plt.show()

## 5. Label Distribution

Check class balance and bounding box statistics before training.

In [ ]:
from collections import Counter

counts = {"train": Counter(), "valid": Counter()}
bbox_areas = {"train": [], "valid": []}

for split in ("train", "valid"):
    lbl_dir = COMPOSITE_DIR / split / "labels"
    for lbl_path in lbl_dir.glob("*.txt"):
        for line in lbl_path.read_text().splitlines():
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            cls_id = int(parts[0])
            w, h = float(parts[3]), float(parts[4])
            counts[split][cls_id] += 1
            bbox_areas[split].append(w * h)

print("Class distribution:")
for split in ("train", "valid"):
    print(f"  {split}:")
    for cls_id in sorted(counts[split]):
        name = CLASS_NAMES[cls_id] if cls_id < len(CLASS_NAMES) else f"class_{cls_id}"
        print(f"    {name:12s}: {counts[split][cls_id]:,}")
    empty = sum(1 for p in (COMPOSITE_DIR / split / "labels").glob("*.txt") if p.stat().st_size == 0)
    print(f"    {'(negatives)':12s}: {empty}")

# Bbox area histogram
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, split in zip(axes, ("train", "valid")):
    ax.hist(bbox_areas[split], bins=50, edgecolor="black", alpha=0.7)
    ax.set_title(f"{split} — bbox area distribution")
    ax.set_xlabel("Normalised area (w × h)")
    ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

## 6. Hyperparameters

All training hyperparameters in one place. Adjust before running the training cell.

In [ ]:
# ── Hyperparameters ───────────────────────────────────────────────────────
EPOCHS    = 40       # total epochs (early stopping will cut short if no improvement)
BATCH     = 16       # Colab T4 (16 GB VRAM) handles batch 16 at imgsz 640
IMGSZ     = 640      # input image size
PATIENCE  = 15       # early stopping: stop after N epochs without mAP improvement
DEVICE    = "cuda"   # always cuda on Colab (NOT mps)
WORKERS   = 2        # dataloader workers (Colab has 2 CPUs)

EXPERIMENT = "whiteboard_composite_v2"

## 7. Train

Loads COCO-pretrained `yolov8n.pt` (NOT the old 5-class checkpoint — the detection head is incompatible with our 2-class task).

Training progress will be printed live. On a T4 GPU with batch 16, expect ~15-25 minutes for 40 epochs on ~1,900 images.

In [ ]:
from ultralytics import YOLO

# Load COCO-pretrained YOLOv8-nano (head will be re-initialised for 2 classes)
model = YOLO("yolov8n.pt")

results = model.train(
    data=str(YAML_PATH.resolve()),
    epochs=EPOCHS,
    patience=PATIENCE,
    imgsz=IMGSZ,
    batch=BATCH,
    workers=WORKERS,
    device=DEVICE,
    name=EXPERIMENT,
    project=str(RUNS_DIR),
    exist_ok=True,
    # Augmentations
    hsv_h=0.01,
    hsv_s=0.30,
    hsv_v=0.40,
    fliplr=0.5,
    degrees=5.0,
    scale=0.5,
    translate=0.1,
    mosaic=1.0,
    erasing=0.4,
    amp=True,
)

## 8. Evaluate

Run validation and report per-class mAP. Review the training curves and confusion matrix.

In [ ]:
# Run validation on the best checkpoint
metrics = model.val(data=str(YAML_PATH.resolve()))

print("=" * 55)
print("VALIDATION RESULTS")
print("=" * 55)
if hasattr(metrics, "box"):
    print(f"  mAP@50 (all)     : {metrics.box.map50:.4f}")
    print(f"  mAP@50:95 (all)  : {metrics.box.map:.4f}")
    if hasattr(metrics.box, "maps") and len(metrics.box.maps) >= 2:
        for i, name in enumerate(CLASS_NAMES):
            print(f"  mAP@50 ({name:10s}): {metrics.box.maps[i]:.4f}")
print("=" * 55)

In [ ]:
# Display training curves (loss, mAP, precision, recall)
from IPython.display import Image, display

curves_dir = RUNS_DIR / EXPERIMENT

for plot_name in ["results.png", "confusion_matrix.png", "P_curve.png", "R_curve.png"]:
    plot_path = curves_dir / plot_name
    if plot_path.exists():
        print(f"\n--- {plot_name} ---")
        display(Image(filename=str(plot_path), width=800))

## 9. Test Predictions

Run the trained model on a few validation images to visually verify detection quality.

In [ ]:
# Run inference on 4 random validation images
best_pt = RUNS_DIR / EXPERIMENT / "weights" / "best.pt"
val_model = YOLO(str(best_pt))

val_imgs = sorted((COMPOSITE_DIR / "valid" / "images").glob("comp_*"))
test_samples = random.sample(val_imgs, min(4, len(val_imgs)))

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, img_path in zip(axes.flat, test_samples):
    img = cv2.imread(str(img_path))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    preds = val_model.predict(source=img_rgb, conf=0.25, verbose=False)
    if preds and len(preds[0].boxes) > 0:
        boxes = preds[0].boxes
        for j in range(len(boxes)):
            x1, y1, x2, y2 = boxes.xyxy[j].int().tolist()
            cls_id = int(boxes.cls[j])
            conf = float(boxes.conf[j])
            color = COLORS[cls_id] if cls_id < len(COLORS) else (128, 128, 128)
            cv2.rectangle(img_rgb, (x1, y1), (x2, y2), color, 2)
            label = f"{CLASS_NAMES[cls_id]} {conf:.2f}"
            cv2.putText(img_rgb, label, (x1, max(y1 - 5, 10)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)

    ax.imshow(img_rgb)
    ax.set_title(img_path.name, fontsize=9)
    ax.axis("off")

plt.suptitle("Validation Predictions — blue=equation, red=whiteboard", fontsize=13)
plt.tight_layout()
plt.show()

## 10. Export & Download

Copy the best checkpoint to Google Drive so it persists after the Colab session ends. You can also download it directly to your local machine.

In [ ]:
# Save best.pt to Google Drive
best_src = RUNS_DIR / EXPERIMENT / "weights" / "best.pt"
drive_dst = DRIVE_OUT / EXPERIMENT / "weights"
drive_dst.mkdir(parents=True, exist_ok=True)

shutil.copy2(best_src, drive_dst / "best.pt")
print(f"Saved to Drive: {drive_dst / 'best.pt'}")

# Also copy key training artifacts
for artifact in ["results.png", "confusion_matrix.png", "args.yaml", "results.csv"]:
    src = RUNS_DIR / EXPERIMENT / artifact
    if src.exists():
        shutil.copy2(src, DRIVE_OUT / EXPERIMENT / artifact)

print(f"Training artifacts saved to: {DRIVE_OUT / EXPERIMENT}")

In [ ]:
# Download best.pt directly to your local machine (browser download)
from google.colab import files
files.download(str(best_src))
# After download, place it at: model/best_v2.pt in your local project

---

# Part 2 — Full Inference Pipeline Demo

Now that we have a trained model, let's run the complete 3-stage pipeline:

1. **Detection** — YOLOv8 finds whiteboard + equation regions
2. **Post-processing** — Merge fragmented bboxes, filter ceiling false positives, keep equations inside whiteboards
3. **OCR** — pix2tex (LaTeX) → TrOCR (handwriting) → PaddleOCR (fallback)

Upload a whiteboard photo to `My Drive/mla-proj/test_images/` or use the validation samples.

## 11. Pipeline Helpers — Detection Post-processing

These functions handle the post-processing between raw YOLO output and OCR input:
- **`filter_by_center_y`** — suppress ceiling false positives (top 25% of image)
- **`merge_nearby_equations`** — combine fragmented equation bboxes into one
- **`equations_inside_whiteboards`** — keep only equations that fall inside a detected whiteboard

In [ ]:
import dataclasses

@dataclasses.dataclass
class CroppedRegion:
    """One detected region from YOLO."""
    class_id:   int
    class_name: str
    confidence: float
    bbox_xyxy:  tuple  # (x1, y1, x2, y2) absolute px
    crop:       np.ndarray  # H×W×C uint8


def yolo_to_regions(results, image, class_names):
    """Convert ultralytics Results to a list of CroppedRegion."""
    regions = []
    if not results or len(results) == 0:
        return regions
    boxes = results[0].boxes
    H, W = image.shape[:2]
    for i in range(len(boxes)):
        cls_id = int(boxes.cls[i].item())
        score = float(boxes.conf[i].item())
        x1, y1, x2, y2 = boxes.xyxy[i].tolist()
        x1, y1 = max(0, int(x1)), max(0, int(y1))
        x2, y2 = min(W, int(x2)), min(H, int(y2))
        crop = image[y1:y2, x1:x2].copy()
        regions.append(CroppedRegion(
            class_id=cls_id,
            class_name=class_names.get(cls_id, f"class_{cls_id}"),
            confidence=score,
            bbox_xyxy=(x1, y1, x2, y2),
            crop=crop,
        ))
    regions.sort(key=lambda r: (r.bbox_xyxy[1], r.bbox_xyxy[0]))
    return regions


def filter_by_center_y(regions, image_height, min_ratio=0.25):
    """Discard detections in the top portion of the image (ceiling lights)."""
    return [r for r in regions
            if (r.bbox_xyxy[1] + r.bbox_xyxy[3]) / 2 > image_height * min_ratio]


def merge_nearby_equations(equations, image, iou_threshold=0.05, gap_ratio=0.03):
    """
    Merge equation bboxes that overlap or are horizontally adjacent.
    YOLO often fragments one handwritten equation into multiple small boxes.
    """
    if len(equations) <= 1:
        return equations

    img_h, img_w = image.shape[:2]
    max_gap = int(img_w * gap_ratio)

    boxes = [list(eq.bbox_xyxy) for eq in equations]
    confs = [eq.confidence for eq in equations]
    merged = [False] * len(boxes)

    changed = True
    while changed:
        changed = False
        for i in range(len(boxes)):
            if merged[i]:
                continue
            for j in range(i + 1, len(boxes)):
                if merged[j]:
                    continue
                bx1, by1, bx2, by2 = boxes[i]
                cx1, cy1, cx2, cy2 = boxes[j]
                # IoU check
                ix1, iy1 = max(bx1, cx1), max(by1, cy1)
                ix2, iy2 = min(bx2, cx2), min(by2, cy2)
                inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
                union = (bx2-bx1)*(by2-by1) + (cx2-cx1)*(cy2-cy1) - inter
                iou = inter / max(union, 1)
                # Horizontal adjacency check
                h_gap = max(0, max(bx1, cx1) - min(bx2, cx2))
                v_overlap = min(by2, cy2) - max(by1, cy1)
                min_h = min(by2 - by1, cy2 - cy1)
                similar_height = v_overlap > 0.3 * max(min_h, 1)

                if iou > iou_threshold or (h_gap < max_gap and similar_height):
                    boxes[i] = [min(bx1,cx1), min(by1,cy1), max(bx2,cx2), max(by2,cy2)]
                    confs[i] = max(confs[i], confs[j])
                    merged[j] = True
                    changed = True

    result = []
    for i in range(len(boxes)):
        if merged[i]:
            continue
        x1, y1, x2, y2 = boxes[i]
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(img_w, x2), min(img_h, y2)
        result.append(CroppedRegion(
            class_id=0, class_name="equation", confidence=confs[i],
            bbox_xyxy=(x1, y1, x2, y2), crop=image[y1:y2, x1:x2].copy(),
        ))
    result.sort(key=lambda r: (r.bbox_xyxy[1], r.bbox_xyxy[0]))
    return result


def equations_inside_whiteboards(equations, whiteboards):
    """Keep only equations whose center falls inside a whiteboard bbox."""
    if not whiteboards:
        return equations
    valid = []
    for eq in equations:
        eq_cx = (eq.bbox_xyxy[0] + eq.bbox_xyxy[2]) / 2
        eq_cy = (eq.bbox_xyxy[1] + eq.bbox_xyxy[3]) / 2
        for wb in whiteboards:
            wx1, wy1, wx2, wy2 = wb.bbox_xyxy
            if wx1 <= eq_cx <= wx2 and wy1 <= eq_cy <= wy2:
                valid.append(eq)
                break
    return valid


V2_CLASS_NAMES = {0: "equation", 1: "whiteboard"}
print("Pipeline helpers loaded.")

## 12. OCR Engine — Math-Specific Preprocessing + Fallback Chain

Preprocessing pipeline for equation crops:
1. Grayscale → CLAHE contrast enhancement → Otsu binarization (if noisy) → Upscale to 320px min height

OCR fallback chain:
1. **pix2tex** (LatexOCR) — outputs LaTeX directly, trained on im2latex
2. **TrOCR** — Microsoft's handwriting transformer, outputs plain text
3. **PaddleOCR** — general-purpose scene text, last resort

In [ ]:
def preprocess_math_crop(crop):
    """
    Math-specific preprocessing for equation crops before OCR.
    Grayscale → CLAHE → conditional Otsu → upscale to 320px min height.
    """
    if crop.size == 0:
        return crop
    gray = cv2.cvtColor(crop, cv2.COLOR_RGB2GRAY)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(gray)
    if np.std(enhanced) > 40:
        _, enhanced = cv2.threshold(enhanced, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    h, w = enhanced.shape[:2]
    if h < 320:
        scale = 320 / h
        enhanced = cv2.resize(enhanced, (int(w * scale), 320), interpolation=cv2.INTER_LINEAR)
    return cv2.cvtColor(enhanced, cv2.COLOR_GRAY2RGB)


class MathOCREngine:
    """Math-aware OCR with fallback chain: pix2tex → TrOCR → PaddleOCR."""

    def __init__(self):
        self._engines = []
        self._init_pix2tex()
        self._init_trocr()
        self._init_paddle()
        if not self._engines:
            raise RuntimeError("No math OCR backend available.")
        print(f"MathOCR backends: {' → '.join(n for n, _ in self._engines)}")

    def _init_pix2tex(self):
        try:
            from pix2tex.cli import LatexOCR
            self._latex_ocr = LatexOCR()
            self._engines.append(("pix2tex", self._run_pix2tex))
            print("  pix2tex loaded")
        except Exception as e:
            print(f"  pix2tex unavailable: {e}")

    def _init_trocr(self):
        try:
            from transformers import TrOCRProcessor, VisionEncoderDecoderModel
            name = "microsoft/trocr-base-handwritten"
            self._trocr_proc = TrOCRProcessor.from_pretrained(name)
            self._trocr_model = VisionEncoderDecoderModel.from_pretrained(name)
            if torch.cuda.is_available():
                self._trocr_model = self._trocr_model.cuda()
            self._engines.append(("trocr", self._run_trocr))
            print("  TrOCR loaded")
        except Exception as e:
            print(f"  TrOCR unavailable: {e}")

    def _init_paddle(self):
        try:
            from paddleocr import PaddleOCR
            self._paddle = PaddleOCR(lang="en", use_textline_orientation=True)
            self._engines.append(("paddleocr", self._run_paddle))
            print("  PaddleOCR loaded")
        except Exception as e:
            print(f"  PaddleOCR unavailable: {e}")

    def recognise(self, crop):
        """Run OCR with fallback. Returns (text, confidence, backend_name)."""
        if crop.size == 0:
            return ("", 0.0, "none")
        preprocessed = preprocess_math_crop(crop)
        for name, fn in self._engines:
            try:
                text, conf = fn(preprocessed)
                if text.strip() and conf > 0.3:
                    return (text.strip(), conf, name)
            except Exception as e:
                print(f"  [{name}] failed: {e}")
        return ("", 0.0, "none")

    def _run_pix2tex(self, crop):
        from PIL import Image as PILImage
        pil_img = PILImage.fromarray(crop)
        latex = self._latex_ocr(pil_img)
        return (latex, 0.85)

    def _run_trocr(self, crop):
        from PIL import Image as PILImage
        pil_img = PILImage.fromarray(crop)
        pixel_values = self._trocr_proc(images=pil_img, return_tensors="pt").pixel_values
        if torch.cuda.is_available():
            pixel_values = pixel_values.cuda()
        with torch.no_grad():
            ids = self._trocr_model.generate(pixel_values, max_new_tokens=128)
        text = self._trocr_proc.batch_decode(ids, skip_special_tokens=True)[0]
        return (text, 0.75)

    def _run_paddle(self, crop):
        bgr = cv2.cvtColor(crop, cv2.COLOR_RGB2BGR)
        results = self._paddle.predict(bgr)
        if not results:
            return ("", 0.0)
        item = results[0]
        texts = item.get("rec_texts", [])
        scores = item.get("rec_scores", [])
        if not texts:
            return ("", 0.0)
        return ("\n".join(texts), sum(scores) / len(scores))


# Initialise the OCR engine (downloads models on first run)
ocr_engine = MathOCREngine()